In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, DateType, TimestampType, StructType, StructField, StringType
from delta.tables import DeltaTable
import uuid
from datetime import datetime

In [0]:
%run ./variables


In [0]:
#watermark 

def get_last_watermark(table_name):
    """Return the last successfully processed _ingest_timestamp for a table.
       Returns epoch (1970-01-01) if no watermark exists yet (first run)."""
    control_exists = spark.catalog.tableExists(Control_table)
    if not control_exists:
        return datetime(1970, 1, 1)
 
    row = (
        spark.table(Control_table)
        .filter(F.col("table_name") == table_name)
        .select("last_watermark")
        .collect()
    )
    if not row:
        return datetime(1970, 1, 1)
    return row[0]["last_watermark"]

In [0]:
def save_watermark(table_name, new_watermark, batch_id):
    """Upsert the watermark for a table into the control table."""
    new_row = spark.createDataFrame(
        [(table_name, new_watermark, batch_id, datetime.now())],
        schema=StructType([
            StructField("table_name",      StringType(),    False),
            StructField("last_watermark",  TimestampType(), False),
            StructField("last_batch_id",   StringType(),    False),
            StructField("last_run_at",     TimestampType(), False),
        ])
    )
 
    control_exists = spark.catalog.tableExists(Control_table)
    if not control_exists:
        new_row.write.format("delta").saveAsTable(Control_table)
    else:
        (
            DeltaTable.forName(spark, Control_table).alias("ctrl")
            .merge(
                new_row.alias("incoming"),
                "ctrl.table_name = incoming.table_name"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    print(f"Watermark saved → {new_watermark} for '{table_name}'")

In [0]:
VALID_CURRENCIES = {"USD", "GBP", "EUR", "AUD"}
VALID_STATUSES   = {"PLACED", "CONFIRMED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"}


#Data Quality rules 
DQ_RULES=[
    ("RO1","order_id must not be null",F.col("order_id").isNotNull()),
    ("RO2","order_line must not be null",F.col("order_line_id").isNotNull()),
    ("RO3","customer_id must not be null",F.col("customer_id").isNotNull()),
    ("RO4","order_date must not be null",F.col("order_date").isNotNull()),
    ("RO5","order_total must be greater than zero",F.col("order_total")>0),
    ("RO6","quantity must be greater than zero",F.col("quantity")>0),
    ("RO7","currency must be a valid ISO code",F.col("currency").isin(list(VALID_CURRENCIES))),
    ("RO8","order_status must not be know enum value",F.col("order_status").isin(list(VALID_STATUSES))),
    ]

def appply_dq_rules(df,rules):
    """
    Adds a dq_failure_reasons column to df.
    Value is null if all rules pass (record is clean).
    Value is a pipe-separated string of failing rule codes if any rule fails.
    e.g. null, "R03", "R05|R07"
    """

    # Add a boolean pass/fail column per rule
    for code, _, condition in rules:
        df=df.withColumn(
            f"_dq_{code}",
            F.when(condition,True).otherwise(False)
        )

    # Build failure reason: array of codes where the rule FAILED
    failure_array=F.array(*[
        F.when(~F.col(f"_dq_{code}"),F.lit(f"{code}:{desc}")) for code,desc, _ in rules
    ])
    
    # Filter nulls from the array (rules that passed return null above)
    # then join into a pipe-separated string

    df=df.withColumn(
        "dq_failure_reasons",F.array_join(F.array_remove(failure_array,None), " | ")
    )
    # Replace empty string with null so IS NULL check works cleanly

    df = df.withColumn(
    "dq_failure_reasons",
    F.when(F.col("dq_failure_reasons") == "", None)
        .otherwise(F.col("dq_failure_reasons"))
    )

    # Drop the intermediate per-rule boolean columns
    rule_cols = [f"_dq_{code}" for code, _, _ in rules]
    df = df.drop(*rule_cols)

    return df 

In [0]:
def silver_transformation(incremental_bronze,batch_id):
    duplicate_dedup=Window.partitionBy("order_id","order_line_id").orderBy(F.col("_ingest_timestamp").desc())

    incoming_df=(incremental_bronze
             .withColumn("order_date",F.col("order_date").cast(DateType()))
             .withColumn("quantity",F.col("quantity").cast(IntegerType()))
             .withColumn("unit_price",F.col("unit_price").cast(DoubleType()))
             .withColumn("order_total",F.col("order_total").cast(DoubleType()))
             .drop("_rescued_date")
             .withColumn("_silver_processed_at",F.current_timestamp())
             .withColumn("_batch_id", F.lit(batch_id))
             .withColumn("_row_num", F.row_number().over(duplicate_dedup))
             .filter(F.col("_row_num") == 1)   # keep only the latest copy
             .drop("_row_num")
             .withColumn("processing_date", F.to_date(F.col("_ingest_timestamp")))
             .withColumn("arrival_delay_days",
                F.datediff(F.col("processing_date"), F.col("order_date")))
             .withColumn("is_late_arrival",(F.col("arrival_delay_days") > 0))
             .withColumn("is_deleted",  F.lit(False))
             .withColumn("deleted_at",         F.lit(None).cast(TimestampType()))
             .withColumn("deleted_by_run_id",  F.lit(None).cast(StringType()))
                               # clean up, no need in final Silver
)
    return appply_dq_rules(incoming_df,DQ_RULES)

    


In [0]:
def merge_into_silver(pass_df, fail_df, batch_id, trigger="normal"):
    """MERGE pass records into Silver, append fail records to quarantine."""
    #spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    fail_count = fail_df.count()
    if fail_count>0:
            (
                fail_df
                .withColumn("dq_detected_at",F.current_timestamp())
                .withColumn("dq_batch_id",F.lit(batch_id))
                .write.format('delta').mode("append").option("mergeSchema","true").saveAsTable(QUARANTINE_TABLE)
            )
            print(f"\nQuarantine: {fail_count} record(s) written.")

            print("\nFailure breakdown:")
            display(
                fail_df
                .groupBy("dq_failure_reasons")
                .count()
                .orderBy(F.col("count").desc())
            )
    else:
        print("No DQ failures this batch.")

    silver_table_exists=spark.catalog.tableExists(Silver_table)

    if not silver_table_exists:
        pass_df.write.format('delta').option("mergeSchema","true").saveAsTable(Silver_table)
        print(f"Silver table created on first run. Rows: {pass_df.count()}")

    else:
        target_silver_table=DeltaTable.forName(spark,Silver_table)

        (target_silver_table.alias("t").merge(pass_df.alias("i"),"t.order_id = i.order_id AND t.order_line_id = i.order_line_id").
         withSchemaEvolution().whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        print(f"Silver MERGE complete [{trigger}]. Total: {spark.table(Silver_table).count()}")

          


In [0]:
def log_backfill(batch_id, scenario, date_start, date_end, rows_reprocessed, notes):
    """Append a record to the backfill log — full audit trail."""
    row = spark.createDataFrame(
        [(batch_id, scenario, str(date_start), str(date_end),
          rows_reprocessed, notes, datetime.now())],
        schema=StructType([
            StructField("batch_id",          StringType(),    False),
            StructField("scenario",          StringType(),    False),
            StructField("date_start",        StringType(),    False),
            StructField("date_end",          StringType(),    False),
            StructField("rows_reprocessed",  IntegerType(),   False),
            StructField("notes",             StringType(),    True),
            StructField("executed_at",       TimestampType(), False),
        ])
    )
    if not spark.catalog.tableExists(BACKFILL_LOG):
        row.write.format("delta").saveAsTable(BACKFILL_LOG)
    else:
        row.write.format("delta").mode("append").saveAsTable(BACKFILL_LOG)
    print(f"Backfill logged: {scenario} | {date_start} → {date_end} | {rows_reprocessed} rows")


In [0]:

#SCENARIO B — Targeted date range reprocess from Bronze

def backfill_date_range(start_date, end_date,notes=""):
    """
    Reprocess Bronze records for a specific order_date range.
    Re-applies full Silver transformation (cast, dedup, DQ, late arrival).
    MERGEs corrected records into Silver and recomputes Gold for those dates.
    Does NOT reset or alter the incremental watermark.
 
    Args:
        start_date : str, "YYYY-MM-DD" — inclusive start of affected range
        end_date   : str, "YYYY-MM-DD" — inclusive end of affected range
        notes      : str, reason for backfill (logged to backfill_log)
    """

    backfill_batch_id=str(uuid.uuid4())
    print(f"Starting backfill: {start_date} → {end_date}")
    print(f"Backfill batch_id: {backfill_batch_id}")

    # Read Bronze filtered by order_date range — bypass the watermark entirely
    # We use order_date (business date), not _ingest_timestamp (pipeline date)

    bronze_df = spark.table(Bronze_table)

    affected_bronze = (
        bronze_df
        .withColumn("order_date_cast", F.col("order_date").cast(DateType()))
        .filter(
            (F.col("order_date_cast") >= F.lit(start_date).cast(DateType())) &
            (F.col("order_date_cast") <= F.lit(end_date).cast(DateType()))
        )
        .drop("order_date_cast")
    )
    affected_count = affected_bronze.count()
    print(f"Bronze rows in date range: {affected_count}")

    if affected_count == 0:
        print("No Bronze records found for this date range. Nothing to backfill.")
        return

    # Apply full Silver transformation with fixed logic
    validated = silver_transformation(affected_bronze, backfill_batch_id)
    pass_df   = (validated.filter(F.col("dq_failure_reasons").isNull())
                          .drop("dq_failure_reasons"))
    fail_df   = validated.filter(F.col("dq_failure_reasons").isNotNull())
 
    pass_count = pass_df.count()
    fail_count = fail_df.count()
    print(f"DQ: {pass_count} pass, {fail_count} fail")
 
    # MERGE corrected records into Silver
    merge_into_silver(pass_df, fail_df, backfill_batch_id, trigger="backfill")
    
    # Log the backfill — mandatory audit trail
    log_backfill(
        batch_id         = backfill_batch_id,
        scenario         = "B: targeted_date_range_reprocess",
        date_start       = start_date,
        date_end         = end_date,
        rows_reprocessed = affected_count,
        notes            = notes or "Targeted date range backfill"
    )
 
    print(f"\nBackfill complete for {start_date} → {end_date}")

In [0]:
#SCENARIO C — Quarantine rescue
def rescue_from_quarantine(rule_code_to_relax, updated_rules, notes=""):
    """
    Re-validate quarantined records with updated DQ rules.
    Records that now pass are MERGEd into Silver.
    Records that still fail remain in quarantine (not touched).
 
    Args:
        rule_code_to_relax : str, e.g. "R07"
        updated_rules      : list of DQ rule tuples with the fix applied
        notes              : reason for rescue
    """
    if not spark.catalog.tableExists(QUARANTINE_TABLE):
        print("No quarantine table found. Nothing to rescue.")
        return
 
    quarantine_df    = spark.table(QUARANTINE_TABLE)
    quarantine_count = quarantine_df.count()
    print(f"Total records in quarantine: {quarantine_count}")
 
    if quarantine_count == 0:
        print("Quarantine is empty. Nothing to rescue.")
        return
 
    # Candidates: records where the only failure was the relaxed rule
    # (If a record failed R07 AND R03, it should stay in quarantine)
    candidates = quarantine_df.filter(
        F.col("dq_failure_reasons").contains(rule_code_to_relax)
    )
 
    rescue_batch_id = str(uuid.uuid4())
    print(f"Rescue batch_id      : {rescue_batch_id}")
    print(f"Quarantine candidates: {candidates.count()}")
 
    # Drop old DQ columns so we can re-validate cleanly
    cols_to_drop = ["dq_failure_reasons", "dq_detected_at", "dq_batch_id"]
    revalidated  = appply_dq_rules(
        candidates.drop(*[c for c in cols_to_drop if c in candidates.columns]),
        updated_rules
    )
 
    now_pass = (revalidated.filter(F.col("dq_failure_reasons").isNull())
                            .drop("dq_failure_reasons")
                            .withColumn("_batch_id", F.lit(rescue_batch_id))
                            .withColumn("_silver_processed_at", F.current_timestamp()))
 
    still_fail = revalidated.filter(F.col("dq_failure_reasons").isNotNull())
 
    pass_count = now_pass.count()
    fail_count = still_fail.count()
    print(f"Now pass: {pass_count}  |  Still fail: {fail_count}")
 
    if pass_count > 0:
        merge_into_silver(now_pass, still_fail, rescue_batch_id, trigger="quarantine_rescue")

 
        log_backfill(
            batch_id         = rescue_batch_id,
            scenario         = f"C: quarantine_rescue (relaxed {rule_code_to_relax})",
            date_start       = "n/a",
            date_end         = "n/a",
            rows_reprocessed = pass_count,
            notes            = notes
        )
    else:
        print("No records rescued — all still fail updated rules.")



In [0]:
def run_reconciliation():
    recon_batch_id = str(uuid.uuid4())
    print(f"Reconciliation run: {recon_batch_id}")
 
    # ── Step 1: files currently in landing
    try:
        current_files = {f.path for f in dbutils.fs.ls(file_landing_path)}
    except Exception:
        current_files = set()
 
    print(f"Files currently in landing : {len(current_files)}")
 
    # ── Step 2: distinct source file paths from active Silver records
    silver_df       = spark.table(Silver_table)
    active_silver   = silver_df.filter(F.col("is_deleted") == False)
    silver_paths    = {row["source_file_name"]
                       for row in active_silver.select("source_file_name").distinct().collect()}
 
    print(f"Source paths in Silver     : {len(silver_paths)}")
 
    # ── Step 3: orphaned paths — in Silver but not in landing
    # Normalise paths: Autoloader may store dbfs:/FileStore/...
    # while dbutils.fs.ls returns dbfs:/FileStore/... too — should match.
    # If not, compare by filename only (last segment of path).
    current_filenames = {p.split("/")[-1] for p in current_files}
    orphaned_paths = [
        p for p in silver_paths
        if p.split("/")[-1] not in current_filenames
    ]
 
    print(f"Orphaned paths (deleted)   : {len(orphaned_paths)}")
    for p in orphaned_paths:
        print(f"  DELETED: {p}")
 
    if not orphaned_paths:
        print("No deletions detected. Silver is in sync with landing.")
        _log_recon(recon_batch_id, 0, 0, "no deletions detected")
        return 0
 
    # ── Step 4: soft-delete Silver records for orphaned source files
    # Build a small dataframe of orphaned paths for the MERGE condition
    orphaned_df = spark.createDataFrame(
        [(p,) for p in orphaned_paths],
        schema=StructType([StructField("orphaned_path", StringType(), False)])
    )
 
    # Join Silver to orphaned list, update is_deleted flag
    (
        DeltaTable.forName(spark, Silver_table).alias("silver")
        .merge(
            orphaned_df.alias("orphaned"),
            # Match on source file path — all records from deleted files
            "silver.source_file_name = orphaned.orphaned_path "
            "AND silver.is_deleted = false"
        )
        .whenMatchedUpdate(set={
            "is_deleted":        "true",
            "deleted_at":        F.current_timestamp(),
            "deleted_by_run_id": F.lit(recon_batch_id),
        })
        .execute()
    )
 
    soft_deleted_count = (
        spark.table(Silver_table)
        .filter(
            (F.col("is_deleted") == True) &
            (F.col("deleted_by_run_id") == recon_batch_id)
        )
        .count()
    )
 
    print(f"\nSoft-deleted {soft_deleted_count} Silver record(s).")
    _log_recon(recon_batch_id, len(orphaned_paths), soft_deleted_count, "soft delete applied")
 
    return soft_deleted_count
 
 
def _log_recon(batch_id, files_deleted, rows_soft_deleted, status):
    row = spark.createDataFrame(
        [(batch_id, files_deleted, rows_soft_deleted, status, datetime.now())],
        schema=StructType([
            StructField("batch_id",           StringType(),  False),
            StructField("files_deleted",      IntegerType(), False),
            StructField("rows_soft_deleted",  IntegerType(), False),
            StructField("status",             StringType(),  False),
            StructField("run_at",             TimestampType(),False),
        ])
    )
    if not spark.catalog.tableExists(RECON_LOG):
        row.write.format("delta").saveAsTable(RECON_LOG)
    else:
        row.write.format("delta").mode("append").saveAsTable(RECON_LOG)

In [0]:
def compute_gold(df):
    return (
        # ── KEY: Gold always filters out soft-deleted records
        df.filter(F.col("is_deleted") == False)
        .groupBy("order_date", "order_status")
        .agg(
            F.count("order_id").alias("order_count"),
            F.round(F.sum("order_total"), 2).alias("gmv"),
            F.round(F.avg("order_total"), 2).alias("avg_order_value"),
            F.sum("quantity").alias("total_units_sold"),
            F.sum(F.col("is_late_arrival").cast(IntegerType())).alias("late_arrival_count"),
        )
        .orderBy("order_date", "order_status")
    )